# 🎭 Speech Emotion Recognition & Sarcasm Detection Demo

This notebook demonstrates the joint **ASR + NLP Sentiment + Vocal Emotion** pipeline developed for our local audio preprocessing project (Topic 3). It shows how we can leverage pre-trained Hugging Face models to transcribe speech, classify vocal emotion, analyze text sentiment, and combine them to detect **sarcasm and passive-aggressive behavior**.

In [ ]:
# Install required libraries if not already installed in your environment
# !pip install transformers torch librosa soundfile pandas matplotlib seaborn

In [ ]:
import os
import json
import torch
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
from pathlib import Path

# Verify device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

## 📁 Step 1: Load and inspect metadata
We have prepared a dataset of emotional audio samples under `data/emotion_samples` with metadata stored in `data/emotion_metadata.json`. Let's load the metadata and pick a sample.

In [ ]:
# Search for metadata in parent dir or current dir
paths_to_try = [Path("../data/emotion_metadata.json"), Path("data/emotion_metadata.json")]
metadata_file = None
for p in paths_to_try:
    if p.exists():
        metadata_file = p
        break

if metadata_file:
    with open(metadata_file, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print(f"Loaded {len(metadata)} emotional samples.")
    df = pd.DataFrame(metadata)
    display(df.head())
else:
    print("Metadata file not found! Please run 'python scripts/download_emotion_samples.py' first.")

## 🔊 Step 2: Load and Visualize an Audio Sample
Let's select an angry statement and plot its waveform and spectrogram.

In [ ]:
if metadata_file and len(metadata) > 0:
    # Select the first sample (should be an angry/sad sample)
    sample_info = metadata[0]
    audio_path = Path(metadata_file.parent / "emotion_samples" / sample_info["file_name"])
    
    # Load audio at 16kHz
    y, sr = librosa.load(audio_path, sr=16000)
    print(f"File: {sample_info['file_name']}")
    print(f"True Emotion: {sample_info['emotion'].upper()}")
    print(f"True Transcription: '{sample_info['transcription']}'")
    
    # Plot waveform
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)
    librosa.display.waveshow(y, sr=sr, color='royalblue')
    plt.title(f"Waveform - {sample_info['emotion'].upper()} voice sample")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    
    # Plot spectrogram
    plt.subplot(2, 1, 2)
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='linear')
    plt.title("Spectrogram")
    plt.colorbar(format='%+2.0f dB')
    plt.tight_layout()
    plt.show()
else:
    print("No samples available to load.")

## 🎭 Step 3: Run Speech Emotion Recognition (SER)
We load the pre-trained vocal emotion model `superb/wav2vec2-base-superb-er` using Hugging Face's pipeline and perform classification.

In [ ]:
print("Initializing Speech Emotion Recognition model...")
emotion_classifier = pipeline(
    "audio-classification",
    model="superb/wav2vec2-base-superb-er",
    device=0 if device == "cuda" else -1
)

if metadata_file:
    # Run prediction
    preds = emotion_classifier(str(audio_path))
    print("\n--- Vocal Emotion Classification Results ---")
    for p in preds:
        print(f"Emotion: {p['label']:8} | Confidence: {p['score']:.2%}")

## 📝 Step 4: Run ASR and Text Sentiment Analysis
Now we transcribe the speech using OpenAI's `whisper-tiny` model and classify the transcribed text's sentiment using `distilbert-base-uncased-finetuned-sst-2-english`.

In [ ]:
print("Initializing Whisper ASR and NLP Sentiment models...")
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=0 if device == "cuda" else -1
)
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if device == "cuda" else -1
)

if metadata_file:
    # 1. Transcribe audio
    asr_res = asr_pipeline(str(audio_path))
    transcribed_text = asr_res["text"]
    print(f"Transcription: \"{transcribed_text}\"")
    
    # 2. Analyze sentiment of the text
    sentiment_res = sentiment_pipeline(transcribed_text)[0]
    print(f"Text Sentiment: {sentiment_res['label']} ({sentiment_res['score']:.2%})")

## 🔄 Step 5: Multi-modal Sarcasm Detection
Sarcasm or passive-aggressiveness is characterized by a mismatch between the **literal meaning of the words** (text sentiment) and the **vocal expression** (voice emotion).

Rules:
- **Literal Positive Words** (POSITIVE sentiment) + **Negative Vocal Tone** (angry, sad) = **Sarcasm Alert** ⚠️
- **Literal Negative Words** (NEGATIVE sentiment) + **Happy Vocal Tone** (happy) = **Sarcasm Alert** ⚠️

In [ ]:
def check_sarcasm(text_sentiment, voice_emotion):
    text_sentiment = text_sentiment.lower()
    voice_emotion = voice_emotion.lower()
    
    is_sarcastic = False
    reason = "Aligned verbal and vocal communication"
    
    if text_sentiment == "positive" and voice_emotion in ["angry", "sad"]:
        is_sarcastic = True
        reason = f"Positive words spoken with a negative voice ({voice_emotion})"
    elif text_sentiment == "negative" and voice_emotion == "happy":
        is_sarcastic = True
        reason = "Negative words spoken with a happy voice"
        
    return is_sarcastic, reason

if metadata_file:
    dominant_voice_emotion = preds[0]["label"]
    is_sarcastic, reason = check_sarcasm(sentiment_res["label"], dominant_voice_emotion)
    
    print("--- SARCASM DETECTION REPORT ---")
    print(f"Text: \"{transcribed_text}\"")
    print(f"Vocal Emotion: {dominant_voice_emotion.upper()}")
    print(f"Text Sentiment: {sentiment_res['label'].upper()}")
    print(f"Result: {'⚠️ SARCASM / PASSIVE-AGGRESSIVE DETECTED!' if is_sarcastic else '✅ Normal Speech Alignment'}")
    print(f"Reason: {reason}")